# An exceptional point in a photonic molecule laser

M. Brandstetter, M. Liertzer, C. Deutsch, P. Klang, J. Schöberl, H. E. Türeci, G. Strasser, K. Unterrainer, S. Rotter:
*Reversing the pump dependence of a laser at an exceptional point*, Nature Communications 5, 4034 (2014), [arXiv:1404.1837](https://arxiv.org/abs/1404.1837)

Two coupled micro-disk quantum cascade lasers emit at 3.2 THz. One disk is pumped above threshold. When the pump of the second disk is increased, the laser first turns **off**, and later on again. The reason is an exceptional point, where two resonances coalesce.

In the paper the resonances were computed with NGSolve in 3D. We set up the two-mode model of the paper, and then the 3D model of the device.

In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

## Two coupled modes

Two levels with frequencies $a, b$, gain or loss $\alpha, \beta$, and coupling $\gamma$:

$$
H = \begin{pmatrix} a + i \alpha & \gamma \\ \gamma & b + i \beta \end{pmatrix}
$$

An eigenvalue above the real axis is a mode above the lasing threshold. Level A has gain, $\alpha = 0.1$. The gain of level B is increased from $\beta = -0.4$ to $\beta = \alpha$, with $\gamma = 0.15$ (Fig. 1a,b of the paper).
For $a = b$ the eigenvalues meet in the exceptional point at $\beta = \alpha - 2 \gamma$:

In [ ]:
alpha, gamma = 0.1, 0.15
betas = np.linspace(-0.4, alpha, 201)

fig, ax = plt.subplots(1, 2, figsize=(10,3.2), sharey=True)
for axi, (a, b) in zip(ax, [(1, 1), (1, 1.002)]):
    ev = np.array([np.linalg.eigvals([[a+1j*alpha, gamma], [gamma, b+1j*beta]]) for beta in betas])
    sc = axi.scatter(ev.real, ev.imag, c=np.stack([betas,betas],axis=1), s=4)
    axi.axhline(0, color="k", ls="--")
    axi.set_title(f"a = {a}, b = {b}"); axi.set_xlabel("Re")
ax[0].set_ylabel("Im")
plt.colorbar(sc, ax=ax, label=r"$\beta$")
plt.show()

## The device in 3D

Two disks of radius 47 µm and height 3.5 µm with a gap of 2 µm, refractive index $n = 3.61$. They sit on a metal ground plane and carry a metal contact on top, both modelled as perfect conductors. We solve

$$
\operatorname{curl} \operatorname{curl} E = n^2 k^2 E
$$

with $H(\operatorname{curl})$ elements and a PML. The plane $y = 0$ through the two disk centers is a symmetry plane, so we mesh one half:

In [ ]:
r, h, gap, nr = 47, 3.5, 2, 3.61       # micrometers
R, Rpml = 150, 210

half = Box((-Rpml,0,0), (Rpml,Rpml,Rpml))
half.faces.name = "symmetry"
half.faces.Min(Z).name = "ground"

def Disk(x, name):
    cyl = Cylinder((x,0,0), Z, r=r, h=h)
    cyl.faces.Max(Z).name = "top"
    cyl.faces.Min(Z).name = "ground"
    disk = cyl * half
    disk.mat(name)
    disk.maxh = 5
    return disk

diskA, diskB = Disk(-(r+gap/2), "A"), Disk(r+gap/2, "B")
air = Sphere((0,0,0), R) * half - diskA - diskB
air.mat("air")
outer = Sphere((0,0,0), Rpml)
outer.faces.name = "outer"
pmlregion = (outer - Sphere((0,0,0), R)) * half
pmlregion.mat("pml")

mesh = Glue([diskA, diskB, air, pmlregion]).GenerateMesh(maxh=30).Curve(3)
mesh.SetPML(pml.Radial(rad=R, alpha=1j, origin=(0,0,0)), "pml")
print ("elements:", mesh.ne)
Draw (mesh, euler_angles=[-60,0,-20]);

The refractive indices of the two disks are parameters, with absorption for $\operatorname{Im} n > 0$ and gain for $\operatorname{Im} n < 0$. Shift-and-invert Arnoldi gives the resonances next to a shift:

In [ ]:
fes = HCurl(mesh, order=2, complex=True, dirichlet="ground|top|outer")
u, v = fes.TnT()
nA, nB = ParameterC(nr), ParameterC(nr)
n = mesh.MaterialCF({"A" : nA, "B" : nB}, default=1)
a = BilinearForm(curl(u)*curl(v)*dx).Assemble()
m = BilinearForm(n**2*u*v*dx)
print ("ndof =", fes.ndof)

def Resonances(ImnA, ImnB, shift, num=6):
    nA.Set(nr+1j*ImnA)
    nB.Set(nr+1j*ImnB)
    m.Assemble()
    vecs = [a.mat.CreateVector() for i in range(num)]
    lam = ArnoldiSolver(a.mat, m.mat, fes.FreeDofs(), vecs, shift=shift**2)
    modes = [(complex(l)**0.5, vec) for l, vec in zip(lam, vecs)]
    return sorted(modes, key = lambda mode : abs(mode[0]-shift))[:2]     # the two next to the shift

Without gain and loss, the mode pair of the device at 3.2 THz:

In [ ]:
THz = 47.71      # c / (2 pi) in THz micrometer

with TaskManager():
    pair = Resonances(0, 0, 0.0668)
for k, vec in pair:
    print (f"k = {k:.7f},  f = {THz*k.real:.4f} THz,  Q = {k.real/(-2*k.imag):.0f}")
k0 = (pair[0][0].real + pair[1][0].real) / 2
split = abs(pair[0][0].real - pair[1][0].real)
print (f"splitting = {1000*THz*split:.2f} GHz     (paper: 3.3 GHz computed, 3.9 GHz measured)")

The vertical field $E_z$ of the two modes in the plane $z = h/2$. They have three radial maxima and the azimuthal number $m = 3$, as identified in the paper:

In [ ]:
xs, ys = np.meshgrid(np.linspace(-2*r-gap/2, 2*r+gap/2, 381)[1:-1], np.linspace(0, r, 96)[:-1])
indisk = (np.minimum(abs(xs-r-gap/2), abs(xs+r+gap/2))**2 + ys**2 < (r-0.2)**2)
pts = mesh(xs[indisk], ys[indisk], h/2)

mode = GridFunction(fes)
fig, ax = plt.subplots(2, 1, figsize=(9,5.5), sharex=True)
for axi, (k, vec) in zip(ax, pair):
    mode.vec.data = vec
    Ez = mode(pts)[:,2]
    Ez = Ez / Ez[np.argmax(abs(Ez))]
    field = np.full(xs.shape, np.nan)
    field[indisk] = Ez.real
    field = np.concatenate([field[:0:-1], field])          # mirror to y < 0
    axi.imshow(field, extent=(xs.min(), xs.max(), -r, r), cmap="RdBu_r", vmin=-1, vmax=1)
    axi.set_title(f"f = {THz*k.real:.4f} THz"); axi.set_aspect(1)
plt.show()

## Reduction to two modes

The pump changes $n$ in the disks only a little. So we project the 3D problem to the space of the two modes $v_1, v_2$:

$$
A_{red} \, x = k^2 \, \big( M_0 + n_A^2 M_A + n_B^2 M_B \big) \, x, \qquad (A_{red})_{ij} = \int \operatorname{curl} v_j \cdot \operatorname{curl} v_i , \quad (M_A)_{ij} = \int_{\text{disk A}} v_j \cdot v_i , \; \ldots
$$

This is the two-mode model from the beginning, now with entries from the 3D fields:

In [ ]:
V = [vec for k, vec in pair]
tmp = a.mat.CreateVector()

def Reduce(mat):
    red = np.zeros((2,2), dtype=complex)
    for j in range(2):
        tmp.data = mat * V[j]
        for i in range(2):
            red[i,j] = InnerProduct(tmp, V[i], conjugate=False)
    return red

Ared = Reduce(a.mat)
MA, MB, M0 = [ Reduce(BilinearForm(u*v*dx(region)).Assemble().mat) for region in ["A", "B", "air|pml"] ]

def Reduced(ImnA, ImnB):
    return np.sqrt(scipy.linalg.eigvals(Ared, M0 + (nr+1j*ImnA)**2 * MA + (nr+1j*ImnB)**2 * MB))

print ("reduced:", Reduced(0,0))

## Pump sweep through the exceptional point

Disk A is pumped, with $\operatorname{Im} n_A = -0.0052$ its mode is just above threshold. Disk B is absorbing at first, then its pump is increased.
The curves come from the reduced model, the red circles are full 3D solves:

In [ ]:
ImnA = -0.0052
ImnB = np.linspace(0.0025, -0.0065, 301)
ks = np.array([Reduced(ImnA, b) for b in ImnB])

check = [0.0025, -0.0007, -0.0030, -0.0065]
with TaskManager():
    full = np.array([[k for k, vec in Resonances(ImnA, b, k0)] for b in check])
print ("largest difference between reduced model and 3D solve:",
       max(min(abs(k-ks[np.argmin(abs(ImnB-b))])) for b, kk in zip(check, full) for k in kk))

fig, ax = plt.subplots(1, 2, figsize=(11,3.5))
sc = ax[0].scatter(THz*ks.real, 1e5*ks.imag, c=np.stack([-ImnB,-ImnB],axis=1), s=4)
ax[0].plot(THz*full.real, 1e5*full.imag, "o", color="red", mfc="none")
ax[0].axhline(0, color="k", ls="--")
ax[0].set_xlabel("frequency (THz)"); ax[0].set_ylabel(r"Im k $\cdot 10^5$"); ax[0].ticklabel_format(useOffset=False)
plt.colorbar(sc, ax=ax[0], label="gain in disk B")
ax[1].plot(-ImnB, 1e5*ks.imag.max(axis=1))
ax[1].plot(-np.array(check), 1e5*full.imag.max(axis=1), "o", color="red", mfc="none")
ax[1].axhline(0, color="k", ls="--")
ax[1].set_xlabel("gain in disk B,  - Im $n_B$"); ax[1].set_ylabel(r"largest Im k $\cdot 10^5$")
plt.tight_layout(); plt.show()

The lasing mode starts above the threshold, drops below it while the pump of disk B goes **up**, and comes back with a shifted frequency.

## Both pumps

The largest $\operatorname{Im} k$ of the mode pair for all combinations of the two pumps, shown where it is above threshold (compare with Fig. 2b,c of the paper). In the upper right part more pump means less light:

In [ ]:
gains = np.linspace(-0.0025, 0.0065, 121)      # - Im n
growth = np.array([[Reduced(-gA, -gB).imag.max() for gB in gains] for gA in gains])

plt.figure(figsize=(5.5,4.2))
plt.pcolormesh(gains, gains, 1e5*np.maximum(growth, 0), shading="nearest")
plt.colorbar(label=r"largest Im k $\cdot 10^5$ above threshold")
plt.xlabel("gain in disk B"); plt.ylabel("gain in disk A")
plt.show()

## Remarks

* the mesh is coarse, to keep the run time below a minute: one eigenvalue solve takes 7 seconds. With order 3 on a finer mesh the pair is at 3.1845 and 3.1885 THz, with a splitting of 4.06 GHz
* the only loss in this model is radiation; the absorption in the metal layers of the device is part of $\operatorname{Im} n$ here
* above threshold the laser is nonlinear (steady-state ab-initio laser theory); the resonances of the linear problem tell where the threshold is